In [0]:
%run /Users/sandysakthivel2005@gmail.com/common/03_Logger

Logger notebook executed successfully


In [0]:
print(logger)

<Logger SocialMediaPipeline (WARNING)>


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver_catalog1.processed;

In [0]:
%sql
SHOW TABLES IN silver_catalog1.processed;

database,tableName,isTemporary
processed,silver_sentiment,false
processed,silver_trends,false
processed,silver_tweets,false
processed,silver_tweets_clean,false
,_sqldf,true


In [0]:
# Databricks notebook source

# MAGIC %run /Users/sandysakthivel2005@gmail.com/common/03_Logger

from pyspark.sql.functions import *

try:

    logger.info("Silver Trends Pipeline Started")
    print("Silver Trends Pipeline Started")

    # ==========================================
    # Read Bronze Streaming Table
    # ==========================================

    bronzeDF = (
        spark.readStream
             .table("bronze_catalog1.raw.bronze_trends")
    )

    logger.info("Bronze Trends Table Read Successfully")
    print("Bronze Trends Table Read Successfully")

    # ==========================================
    # Remove Only Exact Duplicates
    # ==========================================

    silverDF = bronzeDF.dropDuplicates()

    # ==========================================
    # Handle Null Values
    # ==========================================

    silverDF = (
        silverDF
            .fillna({
                "topic_category": "Unknown",
                "country": "Unknown",
                "tweet_volume": 0,
                "mention_count": 0,
                "retweet_count": 0,
                "trend_score": 0,
                "sentiment_index": 0,
                "impressions": 0,
                "engagement_count": 0
            })
    )

    # ==========================================
    # Data Validation
    # ==========================================

    silverDF = (
        silverDF
            .filter(col("trend_timestamp").isNotNull())
            .filter(col("topic_category").isNotNull())
            .filter(col("country").isNotNull())
            .filter(col("tweet_volume") >= 0)
            .filter(col("mention_count") >= 0)
            .filter(col("retweet_count") >= 0)
            .filter(col("trend_score") >= -1)
            .filter(col("trend_score") <= 1)
            .filter(col("sentiment_index") >= 0)
            .filter(col("sentiment_index") <= 1)
            .filter(col("impressions") >= 0)
            .filter(col("engagement_count") >= 0)
    )

    # ==========================================
    # Standardize Text
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("topic_category", upper(trim(col("topic_category"))))
            .withColumn("country", upper(trim(col("country"))))
    )

    # ==========================================
    # Convert Data Types
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("trend_timestamp", to_timestamp(col("trend_timestamp")))
            .withColumn("tweet_volume", col("tweet_volume").cast("int"))
            .withColumn("mention_count", col("mention_count").cast("int"))
            .withColumn("retweet_count", col("retweet_count").cast("int"))
            .withColumn("trend_score", col("trend_score").cast("double"))
            .withColumn("sentiment_index", col("sentiment_index").cast("double"))
            .withColumn("impressions", col("impressions").cast("int"))
            .withColumn("engagement_count", col("engagement_count").cast("int"))
    )

    # ==========================================
    # Standardize Date
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("trend_date", to_date(col("trend_timestamp")))
    )

    # ==========================================
    # Audit Column
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("silver_load_time", current_timestamp())
            .withColumn("pipeline_name", lit("Silver_Trends"))
    )

    logger.info("Silver Trends Transformations Completed Successfully")
    print("Silver Trends Transformations Completed Successfully")

    # ==========================================
    # Write Silver Table
    # ==========================================

    silverQuery = (
        silverDF.writeStream
            .trigger(availableNow=True)
            .format("delta")
            .outputMode("append")
            .option(
                "checkpointLocation",
                "abfss://socialmedia@socialmediaadls001.dfs.core.windows.net/checkpoints/silver_trends"
            )
            .option("mergeSchema", "true")
            .toTable("silver_catalog1.processed.silver_trends")
    )

    silverQuery.awaitTermination()

    logger.info("Silver Trends Loaded Successfully")
    print("Silver Trends Loaded Successfully")

except Exception as e:

    logger.error(f"Silver Trends Pipeline Failed: {str(e)}")
    print(f"Silver Trends Pipeline Failed: {str(e)}")

    raise

Silver Trends Pipeline Started
Bronze Trends Table Read Successfully
Silver Trends Transformations Completed Successfully
Silver Trends Loaded Successfully
